# Phase 1 QAT — pytorch-quantization INT8

Fine-tunes a pretrained FP32 ResNet-18 on ImageNet-100 with per-channel weight quantization and per-tensor activation quantization.

## 1. Imports & path setup

In [11]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader, Subset

ROOT = Path(".").resolve().parent  # quantized_resnets/
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src" / "qat"))

from config import ExperimentConfig
from data import build_imagenet_dataset
from qat.quantize import (
    calibrate,
    get_quantized_model,
    initialize_quant_modules,
    setup_quantization_descriptors,
)
from qat.train_utils import (
    load_checkpoint,
    save_checkpoint,
    train_one_epoch,
    validate,
)

## 2. Config — edit values here

In [12]:
DATA_DIR        = "/home/pf4636/imagenet"          # ImageNet root (train/ and val/)
FP32_CHECKPOINT = str(ROOT / "checkpoints" / "best.pth")  # pretrained FP32 weights
CHECKPOINT_DIR  = str(ROOT / "checkpoints")        # base dir; a run subfolder is created automatically

EPOCHS          = 15
BATCH_SIZE      = 256
LR              = 1e-4
MOMENTUM        = 0.9
WEIGHT_DECAY    = 1e-4
NUM_WORKERS     = 8
NUM_CLASSES     = 100
CALIB_BATCHES   = 32
INPUT_QUANT_BITS = 4   # 1, 2, 4, or 8
SEED            = 42
RESUME          = None  # set to a checkpoint path string to resume, e.g. ".../qat_phase1_epoch_005.pth"

## 3. Reproducibility & device

In [13]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

[Device] cuda


## 4. Checkpoint directory

In [14]:
run_name       = f"resnet18_qat_int8_in{INPUT_QUANT_BITS}b_{device.type}_bs{BATCH_SIZE}"
checkpoint_dir = str(Path(CHECKPOINT_DIR) / run_name)
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"[Checkpoints] {checkpoint_dir}")

[Checkpoints] /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256


## 5. Quantization setup & model

In [15]:
# Must happen before ResNet-18 is instantiated
setup_quantization_descriptors()
initialize_quant_modules()

model = get_quantized_model(FP32_CHECKPOINT, num_classes=NUM_CLASSES).to(device)
print(f"[Model] FP32 weights loaded from {FP32_CHECKPOINT}")

[Model] FP32 weights loaded from /home/pf4636/code/resnet/quantized_resnets/checkpoints/best.pth


## 6. Data loaders

In [16]:
def _subset(dataset, num_classes: int) -> Subset:
    kept    = set(range(num_classes))
    indices = [i for i, (_, label) in enumerate(dataset.samples) if label in kept]
    return Subset(dataset, indices)

cfg = ExperimentConfig(
    imagenet_path=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    input_quant_bits=INPUT_QUANT_BITS,
)

train_ds = build_imagenet_dataset(cfg, "train")
val_ds   = build_imagenet_dataset(cfg, "val")

if NUM_CLASSES < 1000:
    train_ds = _subset(train_ds, NUM_CLASSES)
    val_ds   = _subset(val_ds,   NUM_CLASSES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"[Data] Train: {len(train_ds):,}  Val: {len(val_ds):,}  (num_classes={NUM_CLASSES})")

[data] Filtered to 129395 samples across 100 classes.
[data] Filtered to 5000 samples across 100 classes.
[Data] Train: 129,395  Val: 5,000  (num_classes=100)


## 7. Calibration

In [17]:
if RESUME is None:
    calibrate(model, train_loader, CALIB_BATCHES, device)
else:
    print("[Calibration] Skipped — amax loaded from resume checkpoint")

[Calibration] Collecting stats over 32 batches ...


W0407 20:50:16.939713 136001185673536 tensor_quantizer.py:238] Load calibrated amax, shape=torch.Size([]).
W0407 20:50:16.940425 136001185673536 tensor_quantizer.py:174] Disable MaxCalibrator
W0407 20:50:16.940861 136001185673536 tensor_quantizer.py:238] Load calibrated amax, shape=torch.Size([64, 1, 1, 1]).
W0407 20:50:16.941069 136001185673536 tensor_quantizer.py:174] Disable MaxCalibrator
W0407 20:50:16.941333 136001185673536 tensor_quantizer.py:238] Load calibrated amax, shape=torch.Size([]).
W0407 20:50:16.941622 136001185673536 tensor_quantizer.py:174] Disable MaxCalibrator
W0407 20:50:16.941903 136001185673536 tensor_quantizer.py:238] Load calibrated amax, shape=torch.Size([64, 1, 1, 1]).
W0407 20:50:16.942359 136001185673536 tensor_quantizer.py:174] Disable MaxCalibrator
W0407 20:50:16.942953 136001185673536 tensor_quantizer.py:238] Load calibrated amax, shape=torch.Size([]).
W0407 20:50:16.943236 136001185673536 tensor_quantizer.py:174] Disable MaxCalibrator
W0407 20:50:16.943

[Calibration] Done. amax values loaded, fake-quant active.


## 8. Optimizer, scheduler & scaler

In [18]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(
    model.parameters(),
    lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()

start_epoch = 1
best_acc    = 0.0

if RESUME:
    start_epoch, best_acc = load_checkpoint(RESUME, model, optimizer, scaler, scheduler)
    print(f"[Resume] Epoch {start_epoch}, best_acc={best_acc:.3f}%")

## 9. QAT fine-tuning loop

In [19]:
for epoch in range(start_epoch, EPOCHS + 1):
    lr = optimizer.param_groups[0]["lr"]
    print(f"\n[Epoch {epoch}/{EPOCHS}]  lr={lr:.2e}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch
    )
    val_loss, val_top1, val_top5 = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"  Train  loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val    loss={val_loss:.4f}  top1={val_top1:.2f}%  top5={val_top5:.2f}%")
    print("-" * 60)

    is_best  = val_top1 > best_acc
    best_acc = max(val_top1, best_acc)

    save_checkpoint(
        state={
            "epoch":     epoch,
            "model":     model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler":    scaler.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_acc":  best_acc,
        },
        directory=checkpoint_dir,
        epoch=epoch,
        is_best=is_best,
    )

print(f"\nQAT COMPLETE — Best val top-1: {best_acc:.3f}%")
print(f"Best weights saved to: {checkpoint_dir}/qat_phase1_best.pth")


[Epoch 1/15]  lr=1.00e-04


[val]: 100%|██████████| 20/20 [00:06<00:00,  3.10it/s]


  Train  loss=0.9413  acc=96.93%
  Val    loss=1.5017  top1=78.08%  top5=93.38%
------------------------------------------------------------
  [Checkpoint] Best QAT model → /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth

[Epoch 2/15]  lr=9.89e-05


[val]: 100%|██████████| 20/20 [00:06<00:00,  3.13it/s]


  Train  loss=0.9324  acc=97.03%
  Val    loss=1.4959  top1=78.14%  top5=93.46%
------------------------------------------------------------
  [Checkpoint] Best QAT model → /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth

[Epoch 3/15]  lr=9.57e-05


...

  Train  loss=0.9157  acc=97.47%
  Val    loss=1.4929  top1=78.50%  top5=93.64%
------------------------------------------------------------

[Epoch 14/15]  lr=4.32e-06


[val]: 100%|██████████| 20/20 [00:06<00:00,  3.11it/s]


  Train  loss=0.9159  acc=97.43%
  Val    loss=1.4898  top1=78.44%  top5=93.58%
------------------------------------------------------------

[Epoch 15/15]  lr=1.09e-06


[val]: 100%|██████████| 20/20 [00:06<00:00,  3.10it/s]

  Train  loss=0.9161  acc=97.44%
  Val    loss=1.4902  top1=78.62%  top5=93.72%
------------------------------------------------------------

QAT COMPLETE — Best val top-1: 78.840%
Best weights saved to: /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth
